### GOLD LAYER - AIRCRAFT FLEET SUMMARY

In [0]:
from pyspark.sql import functions as F

# 1. Read Silver table
silver_aircraft = spark.table(
    "workspace.default.silver_aircraft"
)
# 2. Create Gold-level business aggregation

gold_aircraft_fleet_summary = (
    silver_aircraft
    .groupBy(
        "airline",
        "manufacturer",
        "model",
        "status"
    )
    .agg(
        # Number of aircraft
        F.count("aircraft_id").alias("aircraft_count"),

        # Total passenger capacity
        F.sum("capacity").alias("total_capacity"),

        # Average aircraft capacity
        F.round(
            F.avg("capacity"), 2
        ).alias("average_capacity"),

        # Oldest aircraft
        F.min("manufacture_year").alias(
            "oldest_manufacture_year"
        ),

        # Newest aircraft
        F.max("manufacture_year").alias(
            "newest_manufacture_year"
        ),

        # Average manufacture year
        F.round(
            F.avg("manufacture_year"), 2
        ).alias("average_manufacture_year")
    )
)

# 3. Add Gold processing timestamp
gold_aircraft_fleet_summary = (gold_aircraft_fleet_summary.withColumn("gold_processed_timestamp",F.current_timestamp()))

# 4. Write Gold Delta table

gold_aircraft_fleet_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.gold_aircraft_fleet_summary"
    )



### Gold layer — Silver Flights

In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. READ SILVER TABLE
# ============================================================

silver_flights = spark.table(
    "workspace.default.silver_flights"
)


# ============================================================
# 2. CREATE GOLD-LEVEL BUSINESS AGGREGATION
# ============================================================

gold_flight_summary = (
    silver_flights

    .groupBy(
        "airline",
        "status"
    )

    .agg(

        # ----------------------------------------------------
        # Total number of flights
        # ----------------------------------------------------
        F.count("flight_id").alias(
            "flight_count"
        ),

        # ----------------------------------------------------
        # Total passengers
        # ----------------------------------------------------
        F.sum("passengers").alias(
            "total_passengers"
        ),

        # ----------------------------------------------------
        # Average passengers per flight
        # ----------------------------------------------------
        F.round(
            F.avg("passengers"), 2
        ).alias(
            "average_passengers"
        ),

        # ----------------------------------------------------
        # Total distance covered
        # ----------------------------------------------------
        F.sum("distance_km").alias(
            "total_distance_km"
        ),

        # ----------------------------------------------------
        # Average flight distance
        # ----------------------------------------------------
        F.round(
            F.avg("distance_km"), 2
        ).alias(
            "average_distance_km"
        ),

        # ----------------------------------------------------
        # Total flight duration
        # ----------------------------------------------------
        F.sum("duration_minutes").alias(
            "total_duration_minutes"
        ),

        # ----------------------------------------------------
        # Average flight duration
        # ----------------------------------------------------
        F.round(
            F.avg("duration_minutes"), 2
        ).alias(
            "average_duration_minutes"
        ),

        # ----------------------------------------------------
        # Total delay
        # ----------------------------------------------------
        F.sum("delay_minutes").alias(
            "total_delay_minutes"
        ),

        # ----------------------------------------------------
        # Average delay
        # ----------------------------------------------------
        F.round(
            F.avg("delay_minutes"), 2
        ).alias(
            "average_delay_minutes"
        ),

        # ----------------------------------------------------
        # Maximum delay
        # ----------------------------------------------------
        F.max("delay_minutes").alias(
            "maximum_delay_minutes"
        ),

        # ----------------------------------------------------
        # Minimum delay
        # ----------------------------------------------------
        F.min("delay_minutes").alias(
            "minimum_delay_minutes"
        )
    )
)


# ============================================================
# 3. ADD GOLD PROCESSING TIMESTAMP
# ============================================================

gold_flight_summary = (
    gold_flight_summary
    .withColumn(
        "gold_processed_timestamp",
        F.current_timestamp()
    )
)

# 4. WRITE GOLD DELTA TABLE

(
    gold_flight_summary
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.gold_flight_summary"
    )
)




### GOLD LAYER - FLIGHT INCIDENT SUMMARY

In [0]:

from pyspark.sql import functions as F
# 1. Read Silver incident table
silver_incidents = spark.table("workspace.default.silver_flight_incidents")

# 2. Create Gold business aggregation

gold_incident_summary = (
    silver_incidents
    .groupBy(
        "airport",
        "incident_type",
        "severity",
        "resolution_status"
    )
    .agg(

        # Total number of incidents
        F.count("incident_id").alias(
            "total_incidents"
        ),

        # Number of unique flights affected
        F.countDistinct("flight_id").alias(
            "affected_flights"
        ),

        # Number of unique aircraft affected
        F.countDistinct("aircraft_id").alias(
            "affected_aircraft"
        ),

        # Open incidents
        F.sum(
            F.when(
                F.col("resolution_status") == "Open",
                1
            ).otherwise(0)
        ).alias(
            "open_incidents"
        ),

        # Investigating incidents
        F.sum(
            F.when(
                F.col("resolution_status") == "Investigating",
                1
            ).otherwise(0)
        ).alias(
            "investigating_incidents"
        ),

        # Resolved incidents
        F.sum(
            F.when(
                F.col("resolution_status") == "Resolved",
                1
            ).otherwise(0)
        ).alias(
            "resolved_incidents"
        ),

        # Closed incidents
        F.sum(
            F.when(
                F.col("resolution_status") == "Closed",
                1
            ).otherwise(0)
        ).alias(
            "closed_incidents"
        )
    )
)
# 3. Add Gold processing timestamp

gold_incident_summary = (gold_incident_summary.withColumn("gold_processed_timestamp",F.current_timestamp()))

# 4. Write Gold Delta table
gold_incident_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.gold_incident_summary"
    )


### Gold Maintenance Summary

In [0]:
from pyspark.sql import functions as F
# 1. READ SILVER MAINTENANCE TABLE
silver_maintenance = spark.table( "workspace.default.silver_maintenance"
)
# 2. CREATE GOLD BUSINESS AGGREGATION
gold_maintenance_summary = (
    silver_maintenance

    .groupBy(
        "maintenance_type",
        "severity",
        "status"
    )

    .agg(

        # ----------------------------------------------------
        # Total number of maintenance records
        # ----------------------------------------------------
        F.count("maintenance_id").alias(
            "total_maintenance"
        ),

        # ----------------------------------------------------
        # Number of unique aircraft maintained
        # ----------------------------------------------------
        F.countDistinct("aircraft_id").alias(
            "affected_aircraft"
        ),

        # ----------------------------------------------------
        # Total downtime
        # ----------------------------------------------------
        F.sum("downtime_hours").alias(
            "total_downtime_hours"
        ),

        # ----------------------------------------------------
        # Average downtime per maintenance
        # ----------------------------------------------------
        F.round(
            F.avg("downtime_hours"), 2
        ).alias(
            "average_downtime_hours"
        ),

        # ----------------------------------------------------
        # Maximum downtime
        # ----------------------------------------------------
        F.max("downtime_hours").alias(
            "maximum_downtime_hours"
        ),

        # ----------------------------------------------------
        # Minimum downtime
        # ----------------------------------------------------
        F.min("downtime_hours").alias(
            "minimum_downtime_hours"
        ),

        # ----------------------------------------------------
        # Total maintenance cost
        # ----------------------------------------------------
        F.sum("cost_usd").alias(
            "total_maintenance_cost_usd"
        ),

        # ----------------------------------------------------
        # Average maintenance cost
        # ----------------------------------------------------
        F.round(
            F.avg("cost_usd"), 2
        ).alias(
            "average_maintenance_cost_usd"
        ),

        # ----------------------------------------------------
        # Maximum maintenance cost
        # ----------------------------------------------------
        F.max("cost_usd").alias(
            "maximum_maintenance_cost_usd"
        ),

        # ----------------------------------------------------
        # Minimum maintenance cost
        # ----------------------------------------------------
        F.min("cost_usd").alias(
            "minimum_maintenance_cost_usd"
        )
    )
)

# 3. ADD GOLD PROCESSING TIMESTAMP

gold_maintenance_summary = (
    gold_maintenance_summary
    .withColumn(
        "gold_processed_timestamp",
        F.current_timestamp()
    )
)
# 4. WRITE GOLD DELTA TABLE

(
    gold_maintenance_summary
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.gold_maintenance_summary"
    )
)


### Gold Layer — Flight Sensor Data

In [0]:
from pyspark.sql import functions as F


# ============================================================================
# 1. READ SILVER TABLE
# ============================================================================

silver_sensor = spark.table(
    "workspace.default.silver_sensor_data"
)


# ============================================================================
# 2. GOLD AGGREGATION
# ============================================================================

gold_sensor_summary = (
    silver_sensor
    .groupBy(
        "sensor_type",
        "unit",
        "anomaly_status"
    )
    .agg(

        # Total sensor readings
        F.count("sensor_id").alias("total_sensor_readings"),

        # Unique sensors
        F.countDistinct("sensor_id").alias("unique_sensors"),

        # Unique aircraft affected
        F.countDistinct("aircraft_id").alias("affected_aircraft"),

        # Sensor value statistics
        F.round(
            F.avg("sensor_value"), 2
        ).alias("average_sensor_value"),

        F.round(
            F.min("sensor_value"), 2
        ).alias("minimum_sensor_value"),

        F.round(
            F.max("sensor_value"), 2
        ).alias("maximum_sensor_value"),

        F.round(
            F.stddev("sensor_value"), 2
        ).alias("sensor_value_stddev"),

        # Warning readings
        F.sum(
            F.when(
                F.col("anomaly_status") == "Warning",
                1
            ).otherwise(0)
        ).alias("warning_readings"),

        # Critical readings
        F.sum(
            F.when(
                F.col("anomaly_status") == "Critical",
                1
            ).otherwise(0)
        ).alias("critical_readings"),

        # Normal readings
        F.sum(
            F.when(
                F.col("anomaly_status") == "Normal",
                1
            ).otherwise(0)
        ).alias("normal_readings")
    )
)


# ============================================================================
# 3. ADD GOLD PROCESSING TIMESTAMP
# ============================================================================

gold_sensor_summary = (
    gold_sensor_summary
    .withColumn(
        "gold_processed_timestamp",
        F.current_timestamp()
    )
)


# ============================================================================
# 4. WRITE GOLD TABLE
# ============================================================================

(
    gold_sensor_summary
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.gold_sensor_summary"
    )
)


### Gold_cockpit_display_summary

In [0]:
from pyspark.sql import functions as F


# ============================================================================
# 1. READ SILVER TABLE
# ============================================================================

silver_display = spark.table(
    "workspace.default.silver_display_unit"
)


# ============================================================================
# 2. GOLD AGGREGATION
# ============================================================================

gold_display_summary = (
    silver_display
    .groupBy(
        "display_type",
        "screen_resolution",
        "refresh_rate_hz",
        "bus_connection"
    )
    .agg(

        # Total display units
        F.count("display_unit_id").alias("total_display_units"),

        # Unique display units
        F.countDistinct("display_unit_id").alias("unique_display_units"),

        # Average refresh rate
        F.round(
            F.avg("refresh_rate_hz"), 2
        ).alias("average_refresh_rate_hz"),

        # Minimum refresh rate
        F.min("refresh_rate_hz").alias("minimum_refresh_rate_hz"),

        # Maximum refresh rate
        F.max("refresh_rate_hz").alias("maximum_refresh_rate_hz")
    )
)


# ============================================================================
# 3. ADD GOLD PROCESSING TIMESTAMP
# ============================================================================

gold_display_summary = (
    gold_display_summary
    .withColumn(
        "gold_processed_timestamp",
        F.current_timestamp()
    )
)


# ============================================================================
# 4. WRITE GOLD TABLE
# ============================================================================

(
    gold_display_summary
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.gold_display_summary"
    )
)


### Gold_display_telemetry_summary

In [0]:
from pyspark.sql import functions as F


# ============================================================================
# 1. READ SILVER TABLE
# ============================================================================

silver_display_telemetry = spark.table(
    "workspace.default.silver_display_telemetry"
)


# ============================================================================
# 2. GOLD AGGREGATION
# ============================================================================

gold_display_telemetry_summary = (
    silver_display_telemetry
    .groupBy(
        "flight_phase",
        "pfd_autoflight_mode",
        "mfd_active_page"
    )
    .agg(

        # Total telemetry records
        F.count("timestamp").alias("total_telemetry_records"),

        # Time range
        F.min("timestamp").alias("first_timestamp"),
        F.max("timestamp").alias("last_timestamp"),

        # Airspeed KPIs
        F.round(
            F.avg("indicated_airspeed_kts"), 2
        ).alias("average_airspeed_kts"),

        F.round(
            F.min("indicated_airspeed_kts"), 2
        ).alias("minimum_airspeed_kts"),

        F.round(
            F.max("indicated_airspeed_kts"), 2
        ).alias("maximum_airspeed_kts"),

        # Altitude KPIs
        F.round(
            F.avg("altitude_ft"), 2
        ).alias("average_altitude_ft"),

        F.round(
            F.min("altitude_ft"), 2
        ).alias("minimum_altitude_ft"),

        F.round(
            F.max("altitude_ft"), 2
        ).alias("maximum_altitude_ft"),

        # Heading
        F.round(
            F.avg("magnetic_heading_deg"), 2
        ).alias("average_heading_deg"),

        # Pitch attitude
        F.round(
            F.avg("pitch_attitude_deg"), 2
        ).alias("average_pitch_deg"),

        F.round(
            F.max("pitch_attitude_deg"), 2
        ).alias("maximum_pitch_deg"),

        F.round(
            F.min("pitch_attitude_deg"), 2
        ).alias("minimum_pitch_deg"),

        # Roll attitude
        F.round(
            F.avg("roll_attitude_deg"), 2
        ).alias("average_roll_deg"),

        F.round(
            F.max("roll_attitude_deg"), 2
        ).alias("maximum_roll_deg"),

        F.round(
            F.min("roll_attitude_deg"), 2
        ).alias("minimum_roll_deg"),

        # Bus voltage
        F.round(
            F.avg("pfd_bus_voltage_v"), 2
        ).alias("average_bus_voltage_v"),

        F.round(
            F.min("pfd_bus_voltage_v"), 2
        ).alias("minimum_bus_voltage_v"),

        F.round(
            F.max("pfd_bus_voltage_v"), 2
        ).alias("maximum_bus_voltage_v"),

        # Display brightness
        F.round(
            F.avg("display_brightness_pct"), 2
        ).alias("average_display_brightness_pct"),

        F.round(
            F.min("display_brightness_pct"), 2
        ).alias("minimum_display_brightness_pct"),

        F.round(
            F.max("display_brightness_pct"), 2
        ).alias("maximum_display_brightness_pct")
    )
)


# ============================================================================
# 3. ADD GOLD PROCESSING TIMESTAMP
# ============================================================================

gold_display_telemetry_summary = (
    gold_display_telemetry_summary
    .withColumn(
        "gold_processed_timestamp",
        F.current_timestamp()
    )
)


# ============================================================================
# 4. WRITE GOLD TABLE
# ============================================================================

(
    gold_display_telemetry_summary
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.gold_display_telemetry_summary"
    )
)


### Gold_aerospace_telemetry_summary

In [0]:
from pyspark.sql import functions as F


# ============================================================================
# 1. READ SILVER TABLE
# ============================================================================

silver_aerospace_telemetry = spark.table(
    "workspace.default.silver_aerospace_flight_telemetry"
)


# ============================================================================
# 2. GOLD AGGREGATION
# ============================================================================

gold_aerospace_telemetry_summary = (
    silver_aerospace_telemetry
    .groupBy(
        "flight_id",
        "aircraft_tail_num",
        "flight_phase"
    )
    .agg(

        # Total telemetry records
        F.count("timestamp").alias(
            "total_telemetry_records"
        ),

        # Time range
        F.min("timestamp").alias(
            "first_timestamp"
        ),

        F.max("timestamp").alias(
            "last_timestamp"
        ),

        # Altitude KPIs
        F.round(
            F.avg("altitude_ft"), 2
        ).alias(
            "average_altitude_ft"
        ),

        F.round(
            F.min("altitude_ft"), 2
        ).alias(
            "minimum_altitude_ft"
        ),

        F.round(
            F.max("altitude_ft"), 2
        ).alias(
            "maximum_altitude_ft"
        ),

        # Airspeed KPIs
        F.round(
            F.avg("indicated_airspeed_knots"), 2
        ).alias(
            "average_airspeed_knots"
        ),

        F.round(
            F.min("indicated_airspeed_knots"), 2
        ).alias(
            "minimum_airspeed_knots"
        ),

        F.round(
            F.max("indicated_airspeed_knots"), 2
        ).alias(
            "maximum_airspeed_knots"
        ),

        # Heading
        F.round(
            F.avg("heading_deg"), 2
        ).alias(
            "average_heading_deg"
        ),

        F.round(
            F.min("heading_deg"), 2
        ).alias(
            "minimum_heading_deg"
        ),

        F.round(
            F.max("heading_deg"), 2
        ).alias(
            "maximum_heading_deg"
        ),

        # Pitch
        F.round(
            F.avg("pitch_deg"), 2
        ).alias(
            "average_pitch_deg"
        ),

        F.round(
            F.min("pitch_deg"), 2
        ).alias(
            "minimum_pitch_deg"
        ),

        F.round(
            F.max("pitch_deg"), 2
        ).alias(
            "maximum_pitch_deg"
        ),

        # Roll
        F.round(
            F.avg("roll_deg"), 2
        ).alias(
            "average_roll_deg"
        ),

        F.round(
            F.min("roll_deg"), 2
        ).alias(
            "minimum_roll_deg"
        ),

        F.round(
            F.max("roll_deg"), 2
        ).alias(
            "maximum_roll_deg"
        ),

        # PFD Voltage
        F.round(
            F.avg("pfd_voltage_v"), 2
        ).alias(
            "average_pfd_voltage_v"
        ),

        F.round(
            F.min("pfd_voltage_v"), 2
        ).alias(
            "minimum_pfd_voltage_v"
        ),

        F.round(
            F.max("pfd_voltage_v"), 2
        ).alias(
            "maximum_pfd_voltage_v"
        ),

        # MFD Voltage
        F.round(
            F.avg("mfd_voltage_v"), 2
        ).alias(
            "average_mfd_voltage_v"
        ),

        F.round(
            F.min("mfd_voltage_v"), 2
        ).alias(
            "minimum_mfd_voltage_v"
        ),

        F.round(
            F.max("mfd_voltage_v"), 2
        ).alias(
            "maximum_mfd_voltage_v"
        ),

        # Display Alerts
        F.sum(
            F.when(
                F.col("display_alert_flag") == 1,
                1
            ).otherwise(0)
        ).alias(
            "total_display_alerts"
        ),

        F.sum(
            F.when(
                F.col("display_alert_flag") == 0,
                1
            ).otherwise(0)
        ).alias(
            "normal_telemetry_records"
        )
    )
)


# ============================================================================
# 3. ADD GOLD PROCESSING TIMESTAMP
# ============================================================================

gold_aerospace_telemetry_summary = (
    gold_aerospace_telemetry_summary
    .withColumn(
        "gold_processed_timestamp",
        F.current_timestamp()
    )
)


# ============================================================================
# 4. WRITE GOLD TABLE
# ============================================================================

(
    gold_aerospace_telemetry_summary
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.gold_aerospace_flight_telemetry_summary"
    )
)


# ============================================================================
# 5. DISPLAY GOLD DATA
# ============================================================================

display(gold_aerospace_telemetry_summary)